In [ ]:
import pandas as pd
import folium

# --- Configurazione Percorsi ---
veneto_path = "veneto_csv_latest/ARPAV_coastal_latest.csv"
fvg_path = "fvg_csv_latest_by_station/FVG_coastal_latest_AGGREGATED.csv" 

stazioni_totali = []

# 1. Caricamento Dati Veneto (ARPAV)
try:
    df_vn = pd.read_csv(veneto_path)
    vn_unich = df_vn[['station_id', 'sensor_name', 'lat', 'lon']].drop_duplicates(subset=['station_id']).copy()
    # Rinominiamo per uniformare
    vn_unich.rename(columns={'sensor_name': 'nome_stazione'}, inplace=True)
    vn_unich['provider'] = 'Veneto (ARPAV)'
    vn_unich['color'] = 'blue'  # Colore cerchio
    stazioni_totali.append(vn_unich)
    print(f" Veneto: caricate {len(vn_unich)} stazioni costiere.")
except FileNotFoundError:
    print(f"⚠ Attenzione: File Veneto non trovato in {veneto_path}")

# 2. Caricamento Dati FVG (Protezione Civile)
try:
    df_fvg = pd.read_csv(fvg_path)
    # ATTENZIONE: per il FVG usiamo 'station_name', non 'sensor_name'
    fvg_unich = df_fvg[['station_id', 'station_name', 'lat', 'lon']].drop_duplicates(subset=['station_id']).copy()
    # Rinominiamo per uniformare
    fvg_unich.rename(columns={'station_name': 'nome_stazione'}, inplace=True)
    fvg_unich['provider'] = 'Friuli V.G. (DPC)'
    fvg_unich['color'] = 'green' # Colore cerchio
    stazioni_totali.append(fvg_unich)
    print(f" FVG: caricate {len(fvg_unich)} stazioni costiere.")
except FileNotFoundError:
    print(f"⚠ Attenzione: File FVG aggregato non trovato in {fvg_path}")

# --- Creazione Mappa ---
if stazioni_totali:
    all_points = pd.concat(stazioni_totali, ignore_index=True)
    
    centro_lat = all_points['lat'].mean()
    centro_lon = all_points['lon'].mean()
    
    # Mappa base (usa le tile di default che sono più veloci da caricare)
    mappa = folium.Map(location=[centro_lat, centro_lon], zoom_start=9)
    
    # Aggiungiamo i CIRCLE marker (Vettoriali = 100x più veloci)
    for _, row in all_points.iterrows():
        popup_html = f"""
        <div style='font-family: Arial; font-size: 12px;'>
            <b>Stazione:</b> {row['nome_stazione']}<br>
            <b>ID:</b> {row['station_id']}<br>
            <b>Provider:</b> {row['provider']}
        </div>
        """
        
        # CircleMarker disegna la geometria istantaneamente senza scaricare icone
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=6, # Grandezza del cerchio
            color=row['color'], # Colore del bordo
            fill=True,
            fill_color=row['color'], # Colore di riempimento
            fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{row['nome_stazione']} ({row['provider']})"
        ).add_to(mappa)
    
    print("\n🔵 Blu = Veneto (ARPAV)")
    print("🟢 Verde = Friuli V.G. (DPC)")
    
    display(mappa)
else:
    print("Nessun dato disponibile per creare la mappa.")

✅ Veneto: caricate 7 stazioni costiere.
✅ FVG: caricate 17 stazioni costiere.

🔵 Blu = Veneto (ARPAV)
🟢 Verde = Friuli V.G. (DPC)


In [ ]:
import pandas as pd
import folium

# Load the virtual stations CSV
df = pd.read_csv("Adriatic_Balkan_virtual/adriatic_balkan_virtual_latest.csv")

# Keep one row per station (drop sensor duplicates)
df_stations = df.drop_duplicates(subset=['station_id'])
print(f'Total virtual stations: {len(df_stations)}')
print(f'Regions: {df_stations["provincia"].value_counts().to_dict()}')

# Color map by zone
ZONE_COLORS = {
    'Adriatic_Sea': 'blue',
    'Croatia': 'orange',
    'Slovenia': 'green',
    'Montenegro': 'purple',
    'Albania': 'red',
    'Greece': 'darkred',
}

# Map
center = [df_stations['lat'].mean(), df_stations['lon'].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="CartoDB positron")

for _, row in df_stations.iterrows():
    color = ZONE_COLORS.get(row['provincia'], 'gray')
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        tooltip=f"{row['station_id']}<br>({row['lat']}, {row['lon']})<br>{row['provincia']}"
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:10px; border-radius:8px;
     border:1px solid #ccc; font-size:12px;">
<b>Virtual Stations</b><br>
"""
for zone, color in ZONE_COLORS.items():
    legend_html += f'<span style="color:{color};">●</span> {zone}<br>'
legend_html += "</div>"
m.get_root().html.add_child(folium.Element(legend_html))

print(f'Map ready with {len(df_stations)} stations')
display(m)